In [59]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit,
    MultilabelStratifiedKFold
)

from sklearn.pipeline import Pipeline, FeatureUnion

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import (
    ComplementNB,
    MultinomialNB,
    BernoulliNB
)

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix
)

In [60]:

from pathlib import Path
import pandas as pd
import numpy as np
                                          
# 1. Load dataset

DATA_PATH = Path(
    "/home/areeb/Cornell-1A-ai-companion-risk-radar/data/annotated_training_test_set_600_samples.csv"
)

df = pd.read_csv(DATA_PATH)

label_cols = [
    "label_dependence",
    "label_attachment",
    "label_isolation_displacement",
    "label_crisis_response_frustration",
    "label_other_none"
]

# 2. Validate required columns

required_cols = ["review"] + label_cols

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Original dataset size:", len(df))

# 3. Remove rows without review text

df = df.dropna(subset=["review"]).copy()

df["review"] = df["review"].astype(str).str.strip()

# Remove empty review text
df = df[df["review"] != ""].copy()

# 4. Encode annotations

def encode_label(series):

    values = series.astype("string").str.strip().str.lower()

    mapping = {
        "yes": 1,
        "1": 1,
        "true": 1,
        "no": 0,
        "0": 0,
        "false": 0,
        "": 0
    }

    encoded = values.map(mapping)

    unexpected = values.notna() & encoded.isna()

    if unexpected.any():
        raise ValueError(
            f"Unexpected values in {series.name}: "
            f"{values[unexpected].unique().tolist()}"
        )

    return encoded.fillna(0).astype(int)


y = pd.DataFrame({
    col: encode_label(df[col])
    for col in label_cols
}, index=df.index)

# 5. Remove reviews with all five labels negative

keep = y.sum(axis=1) > 0

removed_count = (~keep).sum()

df = df.loc[keep].copy()
y = y.loc[keep].copy()

# 6. Prepare X and reset indices

X = df["review"].astype(str)

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

print("\nDATA CLEANING SUMMARY")
print("                    -")

print("Original reviews:", len(keep))
print("Reviews removed (all five labels blank/negative):", removed_count)
print("Reviews retained:", len(X))

print("\nPositive examples per label:")
display(y.sum().rename("positive_count").to_frame())

print("\nLabel prevalence (%):")
display((y.mean() * 100).round(2).rename("percentage").to_frame())

print("\nReviews with multiple positive labels:", (y.sum(axis=1) > 1).sum())

print("\nReviews with exactly one positive label:", (y.sum(axis=1) == 1).sum())

print("\nRemaining reviews with zero positive labels:", (y.sum(axis=1) == 0).sum())

print("\nDuplicate review texts:", X.duplicated().sum())

print("\nMissing text:", X.isna().sum())

print("\nFinal X shape:", X.shape)
print("Final y shape:", y.shape)


Original dataset size: 600

DATA CLEANING SUMMARY
                    -
Original reviews: 600
Reviews removed (all five labels blank/negative): 0
Reviews retained: 600

Positive examples per label:


,positive_count
label_dependence,37
label_attachment,34
label_isolation_displacement,52
label_crisis_response_frustration,53
label_other_none,427



Label prevalence (%):


,percentage
label_dependence,6.17
label_attachment,5.67
label_isolation_displacement,8.67
label_crisis_response_frustration,8.83
label_other_none,71.17



Reviews with multiple positive labels: 3

Reviews with exactly one positive label: 597

Remaining reviews with zero positive labels: 0

Duplicate review texts: 0

Missing text: 0

Final X shape: (600,)
Final y shape: (600, 5)


In [61]:

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

print("TRAINING REVIEWS:", len(X_train))
print("TESTING REVIEWS:", len(X_test))

split_summary = pd.DataFrame({
    "full_dataset": y.sum(),
    "train_positive": y_train.sum(),
    "test_positive": y_test.sum()
})

split_summary["train_percentage"] = (
    split_summary["train_positive"] /
    split_summary["full_dataset"] * 100
).round(1)

split_summary["test_percentage"] = (
    split_summary["test_positive"] /
    split_summary["full_dataset"] * 100
).round(1)

display(split_summary)

for label in y.columns:
    train_count = int(y_train[label].sum())
    test_count = int(y_test[label].sum())

    print(f"{label}: Train={train_count}, Test={test_count}")

    if train_count == 0 or test_count == 0:
        print(f"WARNING: {label} has no positive examples in one split.")

print("\nAVERAGE LABELS PER REVIEW")
print("Training:", round(y_train.sum(axis=1).mean(), 3))
print("Testing:", round(y_test.sum(axis=1).mean(), 3))

assert (y_train.sum(axis=1) > 0).all()
assert (y_test.sum(axis=1) > 0).all()

print("\nSplit completed successfully.")


TRAINING REVIEWS: 419
TESTING REVIEWS: 181


,full_dataset,train_positive,test_positive,train_percentage,test_percentage
label_dependence,37,26,11,70.3,29.7
label_attachment,34,24,10,70.6,29.4
label_isolation_displacement,52,36,16,69.2,30.8
label_crisis_response_frustration,53,37,16,69.8,30.2
label_other_none,427,299,128,70.0,30.0


label_dependence: Train=26, Test=11
label_attachment: Train=24, Test=10
label_isolation_displacement: Train=36, Test=16
label_crisis_response_frustration: Train=37, Test=16
label_other_none: Train=299, Test=128

AVERAGE LABELS PER REVIEW
Training: 1.007
Testing: 1.0

Split completed successfully.


In [62]:
# Check the training and testing text data

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining text examples:")
display(X_train.head())

print("\nTesting text examples:")
display(X_test.head())

print("\nText data is ready for TF-IDF feature extraction.")

Training reviews: 419
Testing reviews: 181

Training text examples:


0    Its an amazing for every introvert out there. ...
1    it really is changing me i have a lot of probl...
2                               This app is really bad
3    i dont know what you guys did, but the algorit...
4    Perfect app which works great for the purpose ...
Name: review, dtype: str


Testing text examples:


0    This app is pretty great; you don't have to ta...
1    This app talks to you like a real and tries to...
2    This app got me through some crazy things. It'...
3    I believe that I feel better after using the a...
4    During nights when negative thoughts are overw...
Name: review, dtype: str


Text data is ready for TF-IDF feature extraction.


In [63]:

# ============================================================
# CELL 4: LABEL-SPECIFIC NAIVE BAYES MODELS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

                                                         
# 1. Best configurations from Cell 8
                                                         

LABEL_CONFIGS = {
    "label_attachment": {
        "features": "word_plus_char",
        "alpha": 0.1
    },
    "label_crisis_response_frustration": {
        "features": "word_plus_char",
        "alpha": 0.1
    },
    "label_dependence": {
        "features": "expanded_words",
        "alpha": 0.1
    },
    "label_isolation_displacement": {
        "features": "baseline",
        "alpha": 0.1
    },
    "label_other_none": {
        "features": "baseline",
        "alpha": 0.1
    }
}


                                                         
# 2. Build feature extractors
                                                         

def make_features(feature_config):

    if feature_config == "baseline":
        return TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True
        )

    if feature_config == "expanded_words":
        return TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 3),
            min_df=1,
            max_df=0.98,
            sublinear_tf=True
        )

    if feature_config == "word_plus_char":

        word = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 3),
            min_df=1,
            max_df=0.98,
            sublinear_tf=True
        )

        char = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=1,
            sublinear_tf=True
        )

        return FeatureUnion([
            ("word", word),
            ("char", char)
        ])

    raise ValueError(f"Unknown feature configuration: {feature_config}")


                                                         
# 3. Build Naive Bayes pipeline
                                                         

def make_nb_pipeline(alpha, feature_config):

    return Pipeline([
        ("features", make_features(feature_config)),
        ("classifier", BernoulliNB(alpha=alpha))
    ])


                                                         
# 4. Initialize outputs
                                                         

models = {}
thresholds = {}
best_configs = {}

y_probability = pd.DataFrame(
    index=y_test.index,
    columns=label_cols,
    dtype=float
)

y_pred = pd.DataFrame(
    index=y_test.index,
    columns=label_cols,
    dtype=int
)


                                                         
# 5. Train one model per label
                                                         

for label in label_cols:

    print(f"\nProcessing {label}...")

    if label not in LABEL_CONFIGS:
        raise ValueError(f"Missing configuration for {label}")

    config = LABEL_CONFIGS[label]

    feature_config = config["features"]
    alpha = config["alpha"]

    # Train model using the full training set
    final_model = make_nb_pipeline(
        alpha,
        feature_config
    )

    final_model.fit(
        X_train,
        y_train[label]
    )

    # Get out-of-fold probabilities for threshold selection.
    # This keeps threshold selection separate from test labels.
    from sklearn.model_selection import StratifiedKFold

    n_positive = int(y_train[label].sum())
    n_negative = int((y_train[label] == 0).sum())

    n_splits = min(5, n_positive, n_negative)

    if n_splits < 2:
        raise ValueError(f"Insufficient examples for {label}")

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    oof = np.zeros(len(X_train), dtype=float)

    for train_idx, val_idx in cv.split(
        X_train,
        y_train[label]
    ):

        cv_model = make_nb_pipeline(
            alpha,
            feature_config
        )

        cv_model.fit(
            X_train.iloc[train_idx],
            y_train[label].iloc[train_idx]
        )

        oof[val_idx] = cv_model.predict_proba(
            X_train.iloc[val_idx]
        )[:, 1]

    # Cross-validation average precision
    cv_ap = average_precision_score(
        y_train[label],
        oof
    )

    # Threshold based on training-label prevalence
    target_positive_rate = n_positive / len(y_train)

    threshold = float(
        np.quantile(
            oof,
            1 - target_positive_rate
        )
    )

    if threshold <= 0:

        positive_scores = oof[oof > 0]

        if len(positive_scores) > 0:
            threshold = float(np.median(positive_scores))
        else:
            threshold = 1.0

    oof_predictions = (
        oof >= threshold
    ).astype(int)

    cv_precision = precision_score(
        y_train[label],
        oof_predictions,
        zero_division=0
    )

    cv_recall = recall_score(
        y_train[label],
        oof_predictions,
        zero_division=0
    )

    cv_f1 = f1_score(
        y_train[label],
        oof_predictions,
        zero_division=0
    )

    # Predict held-out test set
    probabilities = final_model.predict_proba(
        X_test
    )[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    # Save results
    models[label] = final_model
    thresholds[label] = threshold

    y_probability[label] = probabilities
    y_pred[label] = predictions

    best_configs[label] = {
        "model": "BernoulliNB",
        "features": feature_config,
        "alpha": alpha,
        "CV average precision": cv_ap,
        "CV precision": cv_precision,
        "CV recall": cv_recall,
        "CV F1": cv_f1
    }

    print("Model: BernoulliNB")
    print("Features:", feature_config)
    print("Alpha:", alpha)
    print("CV average precision:", round(cv_ap, 3))
    print("CV precision:", round(cv_precision, 3))
    print("CV recall:", round(cv_recall, 3))
    print("CV F1:", round(cv_f1, 3))
    print("Threshold:", round(threshold, 6))
    print("Test predicted positives:", int(predictions.sum()))


                                                         
# 6. Display final configurations
                                                         

print("\nBEST CONFIGURATION PER LABEL")

display(
    pd.DataFrame(best_configs).T
)


                                                         
# 7. Prediction summary
                                                         

print("\nPREDICTION SUMMARY")

display(
    pd.DataFrame({
        "actual_positive": y_test.sum(),
        "predicted_positive": y_pred.sum(),
        "threshold": pd.Series(thresholds)
    })
)



Processing label_dependence...
Model: BernoulliNB
Features: expanded_words
Alpha: 0.1
CV average precision: 0.241
CV precision: 0.269
CV recall: 0.269
CV F1: 0.269
Threshold: 0.0
Test predicted positives: 12

Processing label_attachment...
Model: BernoulliNB
Features: word_plus_char
Alpha: 0.1
CV average precision: 0.158
CV precision: 0.167
CV recall: 0.167
CV F1: 0.167
Threshold: 0.0
Test predicted positives: 8

Processing label_isolation_displacement...
Model: BernoulliNB
Features: baseline
Alpha: 0.1
CV average precision: 0.327
CV precision: 0.472
CV recall: 0.472
CV F1: 0.472
Threshold: 9.6e-05
Test predicted positives: 17

Processing label_crisis_response_frustration...
Model: BernoulliNB
Features: word_plus_char
Alpha: 0.1
CV average precision: 0.585
CV precision: 0.514
CV recall: 0.514
CV F1: 0.514
Threshold: 0.0
Test predicted positives: 18

Processing label_other_none...
Model: BernoulliNB
Features: baseline
Alpha: 0.1
CV average precision: 0.898
CV precision: 0.826
CV recall

,model,features,alpha,CV average precision,CV precision,CV recall,CV F1
label_dependence,BernoulliNB,expanded_words,0.1,0.241466,0.269231,0.269231,0.269231
label_attachment,BernoulliNB,word_plus_char,0.1,0.158,0.166667,0.166667,0.166667
label_isolation_displacement,BernoulliNB,baseline,0.1,0.327453,0.472222,0.472222,0.472222
label_crisis_response_frustration,BernoulliNB,word_plus_char,0.1,0.584626,0.513514,0.513514,0.513514
label_other_none,BernoulliNB,baseline,0.1,0.89765,0.826087,0.826087,0.826087



PREDICTION SUMMARY


,actual_positive,predicted_positive,threshold
label_dependence,11,12,2.443389e-41
label_attachment,10,8,1.637283e-54
label_isolation_displacement,16,17,9.648902e-05
label_crisis_response_frustration,16,18,2.348261e-44
label_other_none,128,122,9.952608e-01


In [64]:

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

label_results = []

for label in label_cols:

    print("\n" + "=" * 75)
    print(label)
    print("=" * 75)

    print(classification_report(
        y_test[label],
        y_pred[label],
        labels=[0, 1],
        target_names=["No", "Yes"],
        zero_division=0
    ))

    label_results.append({
        "label": label,
        "precision": precision_score(
            y_test[label],
            y_pred[label],
            zero_division=0
        ),
        "recall": recall_score(
            y_test[label],
            y_pred[label],
            zero_division=0
        ),
        "f1_score": f1_score(
            y_test[label],
            y_pred[label],
            zero_division=0
        ),
        "average_precision": average_precision_score(
            y_test[label],
            y_probability[label]
        ),
        "actual_positive": int(y_test[label].sum()),
        "predicted_positive": int(y_pred[label].sum())
    })

label_metrics = pd.DataFrame(label_results)

print("\nPER-LABEL PERFORMANCE SUMMARY")
display(label_metrics.round(3))


overall_metrics = pd.DataFrame({
    "metric": [
        "Subset accuracy (exact match)",
        "Hamming loss",
        "Micro precision",
        "Micro recall",
        "Micro F1",
        "Macro precision",
        "Macro recall",
        "Macro F1"
    ],
    "score": [
        accuracy_score(y_test, y_pred),
        hamming_loss(y_test, y_pred),
        precision_score(
            y_test, y_pred,
            average="micro",
            zero_division=0
        ),
        recall_score(
            y_test, y_pred,
            average="micro",
            zero_division=0
        ),
        f1_score(
            y_test, y_pred,
            average="micro",
            zero_division=0
        ),
        precision_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        ),
        recall_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        ),
        f1_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        )
    ]
})

print("\nOVERALL MULTILABEL PERFORMANCE")
display(overall_metrics.round(3))


print("\nCONFUSION MATRICES")

for label in label_cols:

    print(f"\n{label}")

    cm = confusion_matrix(
        y_test[label],
        y_pred[label],
        labels=[0, 1]
    )

    display(pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"]
    ))



label_dependence
              precision    recall  f1-score   support

          No       0.95      0.94      0.94       170
         Yes       0.17      0.18      0.17        11

    accuracy                           0.90       181
   macro avg       0.56      0.56      0.56       181
weighted avg       0.90      0.90      0.90       181


label_attachment
              precision    recall  f1-score   support

          No       0.97      0.98      0.97       171
         Yes       0.50      0.40      0.44        10

    accuracy                           0.94       181
   macro avg       0.73      0.69      0.71       181
weighted avg       0.94      0.94      0.94       181


label_isolation_displacement
              precision    recall  f1-score   support

          No       0.94      0.93      0.94       165
         Yes       0.35      0.38      0.36        16

    accuracy                           0.88       181
   macro avg       0.65      0.65      0.65       181
weighted

,label,precision,recall,f1_score,average_precision,actual_positive,predicted_positive
0,label_dependence,0.167,0.182,0.174,0.294,11,12
1,label_attachment,0.500,0.400,0.444,0.345,10,8
2,label_isolation_displacement,0.353,0.375,0.364,0.345,16,17
3,label_crisis_response_frustration,0.667,0.750,0.706,0.621,16,18
4,label_other_none,0.820,0.781,0.800,0.856,128,122



OVERALL MULTILABEL PERFORMANCE


,metric,score
0,Subset accuracy (exact match),0.619
1,Hamming loss,0.122
2,Micro precision,0.701
3,Micro recall,0.685
4,Micro F1,0.693
5,Macro precision,0.501
6,Macro recall,0.498
7,Macro F1,0.498



CONFUSION MATRICES

label_dependence


,Predicted No,Predicted Yes
Actual No,160,10
Actual Yes,9,2


... (4 outputs left out)


label_crisis_response_frustration


,Predicted No,Predicted Yes
Actual No,159,6
Actual Yes,4,12



label_other_none


,Predicted No,Predicted Yes
Actual No,31,22
Actual Yes,28,100


In [ ]:


import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import (
    ComplementNB,
    MultinomialNB,
    BernoulliNB
)
from sklearn.metrics import average_precision_score


                                                                                
# 1. Experiment configurations
                                                                                

NB_MODELS = {
    "ComplementNB": ComplementNB,
    "MultinomialNB": MultinomialNB,
    "BernoulliNB": BernoulliNB
}

ALPHA_GRID = [0.1, 0.5, 1.0]

FEATURE_CONFIGS = {
    "baseline": {
        "word_ngram": (1, 2),
        "word_min_df": 2,
        "char_ngram": None
    },
    "expanded_words": {
        "word_ngram": (1, 3),
        "word_min_df": 1,
        "char_ngram": None
    },
    "word_plus_char": {
        "word_ngram": (1, 3),
        "word_min_df": 1,
        "char_ngram": (2, 5)
    }
}


                                                                                
# 2. Build TF-IDF feature extractors
                                                                                

def make_experiment_features(config):

    word_features = TfidfVectorizer(
        lowercase=True,
        ngram_range=config["word_ngram"],
        min_df=config["word_min_df"],
        max_df=0.98,
        sublinear_tf=True
    )

    if config["char_ngram"] is None:
        return word_features

    char_features = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=config["char_ngram"],
        min_df=1,
        sublinear_tf=True
    )

    return FeatureUnion([
        ("word", word_features),
        ("char", char_features)
    ])


def make_experiment_pipeline(model_name, alpha, config):

    return Pipeline([
        ("features", make_experiment_features(config)),
        ("classifier", NB_MODELS[model_name](alpha=alpha))
    ])


                                                                                
# 3. Run cross-validation for every label
                                                                                

experiment_results = []

for label in label_cols:

    print(f"\nTesting features for: {label}")

    positives = int(y_train[label].sum())
    negatives = int((y_train[label] == 0).sum())

    n_splits = min(3, positives, negatives)

    if n_splits < 2:
        print(f"Skipping {label}: insufficient examples.")
        continue

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for config_name, config in FEATURE_CONFIGS.items():

        for model_name in NB_MODELS:

            for alpha in ALPHA_GRID:

                oof_probabilities = np.zeros(len(X_train))

                for train_idx, val_idx in cv.split(
                    X_train,
                    y_train[label]
                ):

                    model = make_experiment_pipeline(
                        model_name,
                        alpha,
                        config
                    )

                    model.fit(
                        X_train.iloc[train_idx],
                        y_train[label].iloc[train_idx]
                    )

                    oof_probabilities[val_idx] = (
                        model.predict_proba(
                            X_train.iloc[val_idx]
                        )[:, 1]
                    )

                ap = average_precision_score(
                    y_train[label],
                    oof_probabilities
                )

                experiment_results.append({
                    "label": label,
                    "features": config_name,
                    "model": model_name,
                    "alpha": alpha,
                    "average_precision": ap
                })

    print(f"Completed: {label}")


                                                                                
# 4. Display results
                                                                                

experiment_df = pd.DataFrame(experiment_results)

experiment_df = experiment_df.sort_values(
    ["label", "average_precision"],
    ascending=[True, False]
)

print("\nBEST CONFIGURATION FOR EACH LABEL")

best_experiment = (
    experiment_df
    .groupby("label", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_experiment)


print("\nAVERAGE PRECISION BY FEATURE CONFIGURATION")

feature_comparison = (
    experiment_df
    .groupby("features")["average_precision"]
    .agg(["mean", "std"])
    .sort_values("mean", ascending=False)
)

display(feature_comparison)


print("\nTOP 5 CONFIGURATIONS FOR EACH LABEL")

display(
    experiment_df
    .groupby("label", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)



Testing features for: label_dependence
Completed: label_dependence

Testing features for: label_attachment
Completed: label_attachment

Testing features for: label_isolation_displacement


In [ ]:

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)
from scipy.sparse import hstack, csr_matrix
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

EMB_WEIGHT = 1.0
NB_ALPHA = 0.5

feature_types = [
    "TF-IDF only",
    "Embeddings only",
    "TF-IDF + Embeddings"
]

encoder = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating Sentence Transformer embeddings...")

E_train = encoder.encode(
    X_train.tolist(),
    show_progress_bar=True
)

mskf = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = {
    feature: pd.DataFrame(
        0.0,
        index=y_train.index,
        columns=label_cols
    )
    for feature in feature_types
}

for fold, (tr, val) in enumerate(
    mskf.split(X_train, y_train),
    start=1
):

    print(f"\nProcessing fold {fold}/5")

    X_fold_train = X_train.iloc[tr]
    X_fold_val = X_train.iloc[val]

    y_fold_train = y_train.iloc[tr]

    features = FeatureUnion([
        (
            "word",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                sublinear_tf=True
            )
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True
            )
        )
    ])

    T_train = features.fit_transform(X_fold_train)
    T_val = features.transform(X_fold_val)

    scaler = MinMaxScaler().fit(E_train[tr])

    E_tr = normalize(
        scaler.transform(E_train[tr])
    )

    E_val = normalize(
        np.clip(
            scaler.transform(E_train[val]),
            0,
            1
        )
    )

    E_tr_sparse = csr_matrix(E_tr)
    E_val_sparse = csr_matrix(E_val)

    combined_train = hstack([
        T_train,
        EMB_WEIGHT * E_tr_sparse
    ]).tocsr()

    combined_val = hstack([
        T_val,
        EMB_WEIGHT * E_val_sparse
    ]).tocsr()

    feature_matrices = {
        "TF-IDF only": (T_train, T_val),
        "Embeddings only": (E_tr_sparse, E_val_sparse),
        "TF-IDF + Embeddings": (
            combined_train,
            combined_val
        )
    }

    for feature_name, (F_train, F_val) in feature_matrices.items():

        for label in label_cols:

            model = ComplementNB(
                alpha=NB_ALPHA
            )

            model.fit(
                F_train,
                y_fold_train[label]
            )

            probabilities = model.predict_proba(
                F_val
            )[:, 1]

            oof_predictions[feature_name].iloc[val,
                oof_predictions[feature_name].columns.get_loc(label)
            ] = probabilities


comparison_results = []

for feature_name in feature_types:

    for label in label_cols:

        actual = y_train[label]
        probabilities = oof_predictions[feature_name][label]

        precision, recall, thresholds_cv = precision_recall_curve(
            actual,
            probabilities
        )

        f1_values = (
            2 * precision * recall /
            (precision + recall + 1e-9)
        )

        best_f1 = f1_values[:-1].max()

        comparison_results.append({
            "feature_type": feature_name,
            "label": label,
            "average_precision": average_precision_score(
                actual,
                probabilities
            ),
            "best_f1": best_f1,
            "base_rate": actual.mean()
        })


comparison_df = pd.DataFrame(comparison_results)

print("\nFEATURE COMPARISON BY LABEL")
display(
    comparison_df.sort_values(
        ["label", "average_precision"],
        ascending=[True, False]
    ).round(3)
)

print("\nAVERAGE PERFORMANCE BY FEATURE TYPE")
feature_summary = comparison_df.groupby(
    "feature_type"
)[
    ["average_precision", "best_f1"]
].mean().sort_values(
    "average_precision",
    ascending=False
)

display(feature_summary.round(3))

print("\nBEST FEATURE TYPE FOR EACH LABEL")

best_features = (
    comparison_df.sort_values(
        "average_precision",
        ascending=False
    )
    .groupby("label")
    .first()
)

display(best_features.round(3))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 429.30it/s]


Generating Sentence Transformer embeddings...


Batches: 100%|██████████| 14/14 [00:14<00:00,  1.01s/it]



Processing fold 1/5

Processing fold 2/5

Processing fold 3/5

Processing fold 4/5

Processing fold 5/5

FEATURE COMPARISON BY LABEL


,feature_type,label,average_precision,best_f1,base_rate
6,Embeddings only,label_attachment,0.267,0.329,0.057
1,TF-IDF only,label_attachment,0.105,0.189,0.057
11,TF-IDF + Embeddings,label_attachment,0.102,0.200,0.057
13,TF-IDF + Embeddings,label_crisis_response_frustration,0.606,0.571,0.088
8,Embeddings only,label_crisis_response_frustration,0.453,0.500,0.088
3,TF-IDF only,label_crisis_response_frustration,0.251,0.328,0.088
5,Embeddings only,label_dependence,0.165,0.291,0.062
10,TF-IDF + Embeddings,label_dependence,0.102,0.174,0.062
0,TF-IDF only,label_dependence,0.054,0.122,0.062
7,Embeddings only,label_isolation_displacement,0.367,0.460,0.086



AVERAGE PERFORMANCE BY FEATURE TYPE


,average_precision,best_f1
feature_type,,
Embeddings only,0.426,0.488
TF-IDF + Embeddings,0.359,0.397
TF-IDF only,0.266,0.328



BEST FEATURE TYPE FOR EACH LABEL


,feature_type,average_precision,best_f1,base_rate
label,,,,
label_attachment,Embeddings only,0.267,0.329,0.057
label_crisis_response_frustration,TF-IDF + Embeddings,0.606,0.571,0.088
label_dependence,Embeddings only,0.165,0.291,0.062
label_isolation_displacement,Embeddings only,0.367,0.460,0.086
label_other_none,TF-IDF + Embeddings,0.895,0.853,0.714


In [ ]:


oof_lr = pd.DataFrame(0.0, index=Y.index, columns=risk_labels)
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr, te in mskf.split(E, Y):
    for label in risk_labels:
        lr = LogisticRegression(class_weight="balanced", max_iter=2000).fit(E[tr], Y.iloc[tr][label])
        oof_lr.iloc[te, oof_lr.columns.get_loc(label)] = lr.predict_proba(E[te])[:, 1]

for l in risk_labels:
    print(f"{l:38s} base {Y[l].mean():.3f}  LR AP {average_precision_score(Y[l], oof_lr[l]):.3f}")

label_dependence                       base 0.062  LR AP 0.182
label_attachment                       base 0.057  LR AP 0.448
label_isolation_displacement           base 0.087  LR AP 0.380
label_crisis_response_frustration      base 0.088  LR AP 0.630


In [ ]:

for label in label_cols:

    print("\n", label)

    print("Actual positives:", int(y_test[label].sum()))
    print("Predicted positives:", int(y_pred[label].sum()))

    print("Threshold:", round(thresholds[label], 5))

    print("Confusion matrix:")
    display(pd.DataFrame(
        confusion_matrix(
            y_test[label],
            y_pred[label],
            labels=[0, 1]
        ),
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"]
    ))



 label_dependence
Actual positives: 11
Predicted positives: 154
Threshold: 0.0
Confusion matrix:


,Predicted No,Predicted Yes
Actual No,27,143
Actual Yes,0,11



 label_attachment
Actual positives: 10
Predicted positives: 76
Threshold: 0.12113
Confusion matrix:


,Predicted No,Predicted Yes
Actual No,104,67
Actual Yes,1,9



 label_isolation_displacement
Actual positives: 16
Predicted positives: 104
Threshold: 0.0
Confusion matrix:


,Predicted No,Predicted Yes
Actual No,76,89
Actual Yes,1,15



 label_crisis_response_frustration
Actual positives: 16
Predicted positives: 41
Threshold: 0.0
Confusion matrix:


,Predicted No,Predicted Yes
Actual No,138,27
Actual Yes,2,14



 label_other_none
Actual positives: 128
Predicted positives: 143
Threshold: 0.13894
Confusion matrix:


,Predicted No,Predicted Yes
Actual No,24,29
Actual Yes,14,114
